# 1 Tone measurements -- 'CLI'

- All logic lives in the `qlab` package next to this notebook. Each cell is
**one measurement = one call**. Each call drops a timestamped CSV + PNG
into the dated data tree (in config) (see `README.md`).

- Everything here talks **straight to the N5222B over SCPI** —
`our code -> pyvisa -> VNA`. No PycQED and no QCoDeS anywhere: the
instrument is controlled and read directly, and the data comes straight back
to this PC.

- No signal generator connection yet (2 tone scans).

In [1]:
#set up --> change data output path in config
import os, sys
%matplotlib inline

#check in right folder
HERE = os.getcwd()
if not os.path.isdir(os.path.join(HERE, 'qlab')):
    LIB = r''      
    if LIB:        
        os.chdir(LIB); HERE = LIB
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import qlab
st = qlab.connect_scpi()          

[qlab] direct-SCPI station: 'CH1_S11_1' on the N5222B at USB0::0x2A8D::0x2A01::MY58421887::0::INSTR.  Data -> C:/Data_HH/stub-s3


## Resonator spectroscopy 
- recreates source notebook scans through direct scpi communication --> no imported packages.
- Timing data is present for dev --> not necessary to run the scan

In [ ]:
# Low-power S21 scan  (QLab_tra cell 10)
import time

npts, avg = 2001, 200
t0 = time.perf_counter()
r = qlab.resonator_scan(st, center=8.01225e9, span=20e6, power=-30,
                        if_bandwidth=2000, npts=npts, averages=avg,
                        delay_t=6.5e-8, measure='S21')
dt = time.perf_counter() - t0
swp = st.vna.sweep_time()      # the instrument's own estimate for that sweep

print('saved:', r['csv_path'])
print(f'wall clock   : {dt:7.1f} s   ({dt/60:.2f} min)')
print(f'  of which   : {swp:7.1f} s   sweeping (the instrument, irreducible)')
print(f'  overhead   : {dt - swp:7.1f} s   transfer + CSV + plot (ours to shrink)')
print(f'per point    : {dt/npts*1e3:7.2f} ms')
print(f'per pt-avg   : {dt/(npts*avg)*1e6:7.1f} us  <- the rate to compare across settings')

In [ ]:
# S11 reflection  (QLab_tra cell 17)
import time

npts, avg = 2001, 100
t0 = time.perf_counter()
r = qlab.resonator_scan(st, center=8.22025e9, span=50e6, power=-30,
                        if_bandwidth=1000, npts=npts, averages=avg,
                        delay_t=qlab.config.DEFAULT_DELAY_S11, measure='S11')
dt = time.perf_counter() - t0
swp = st.vna.sweep_time()

print('saved:', r['csv_path'])
print(f'wall clock   : {dt:7.1f} s   ({dt/60:.2f} min)')
print(f'  of which   : {swp:7.1f} s   sweeping (the instrument, irreducible)')
print(f'  overhead   : {dt - swp:7.1f} s   transfer + CSV + plot (ours to shrink)')
print(f'per point    : {dt/npts*1e3:7.2f} ms')
print(f'per pt-avg   : {dt/(npts*avg)*1e6:7.1f} us  <- the rate to compare across settings')

In [ ]:
# Punch-out: power sweep, dressed -> bare cavity  (QLab_tra cell 12)
#
# If must be stopped --> interrupt kernel over killing 
# If a read was in flight when you stopped it, run qlab.reset_link() before the next scan.
results = qlab.resonator_power_sweep(st, center=6.5e9, span=1e9,
                                     power_start=-30, power_stop=20, power_step=1,
                                     if_bandwidth=1000, npts=1001, averages=300,
                                     delay_t=6.5e-8, close_fig=True)
print(len(results), 'scans saved')

In [ ]:
# Long-term stability: rescan every 2 h  (QLab_tra cell 14)
# n_runs=None runs forever; set a number for a bounded test.
qlab.stability_monitor(st, center=8.25091e9, span=50e6, interval_s=7200, n_runs=3,
                       power=-30, if_bandwidth=1000, npts=2001, averages=100,
                       delay_t=6.24e-8, close_fig=True)

## 3. Shutdown

Two separate things, and the second is the one that used to be missing:

- `all_off(st)` — RF off, screen handed back to the front panel. Tidy-up.
- `disconnect(st)` — **closes the VISA session and releases the USB claim.**

USBTMC allows exactly one session per device, and that claim lives until the
owning *process* exits. Skipping `disconnect` is why a later `connect_scpi()`
hangs even after you shut every kernel down — Jupyter cannot kill a kernel
that is blocked inside a VISA read, so a zombie survives holding the device.

In [ ]:
qlab.all_off(st)      # VNA RF off, screen back to the front panel
qlab.disconnect(st)   # close the session, release the USB claim  <- do not skip

### Snapshot whatever the VNA is showing right now

**Read-only** 
Saves CSV + PNG into the dated tree like any other measurement.

The CSV carries a `screen_FDATA` column — the instrument's *own* formatted
values — next to our decode of SDATA. If those two agree, the data path is
confirmed end to end.

In [ ]:
snap = qlab.read_trace(st, name='vna_snapshot')

# Cross-check against a marker you've placed on the VNA screen:
import numpy as np
fq = 8.0122e9                      # <- the frequency your marker sits on
i  = np.argmin(np.abs(snap['freq_Hz'] - fq))
print(f"our value at {snap['freq_Hz'][i]/1e9:.6f} GHz = "
      f"{snap['amplitude_dB'][i]:.2f} dB   <-- compare to the marker readout")